# 英語の共参照解析をして代名詞を先行詞に置き換える (Stanza)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/20_semantics/coref_en.ipynb)

NLP クックブック 20-04 ／ 意味解析

英文中の `he` `it` `the company` が何を指しているかを解決したい。情報抽出や要約の前処理として、代名詞を具体的な名前に置き換えたい。

In [1]:
!pip install -q stanza

In [2]:
import stanza

stanza.download("en", processors="tokenize,coref", verbose=False)
nlp = stanza.Pipeline("en", processors="tokenize,coref", verbose=False)   # 多言語の XLM-R large を使うモデル (約 2GB)

text = ("Barack Obama was born in Hawaii. He was elected president in 2008, and his wife Michelle joined him in Washington. "
        "Apple unveiled a new phone on Tuesday. The company said it would go on sale next month.")
doc = nlp(text)

for chain in doc.coref:
    mentions = [" ".join(w.text for w in doc.sentences[m.sentence].words[m.start_word:m.end_word]) for m in chain.mentions]
    if len(mentions) > 1:
        print(f"{chain.representative_text!r:16} ← {mentions}")

'Barack Obama'   ← ['Barack Obama', 'He', 'his', 'him']
'The company'    ← ['Apple', 'The company']
'a new phone'    ← ['a new phone', 'it']


In [3]:
PRONOUNS = {"he", "him", "his", "she", "her", "hers", "it", "its", "they", "them", "their"}
POSSESSIVE = {"his", "her", "its", "their"}

def resolve(doc):
    """代名詞の言及を、その連鎖の代表的な言及 (最も具体的な表現) に置き換える"""
    replace = {}
    for chain in doc.coref:
        for m in chain.mentions:
            words = doc.sentences[m.sentence].words[m.start_word:m.end_word]
            if len(words) == 1 and words[0].text.lower() in PRONOUNS:
                suffix = "'s" if words[0].text.lower() in POSSESSIVE else ""
                replace[(m.sentence, m.start_word)] = chain.representative_text + suffix
    out = []
    for si, sent in enumerate(doc.sentences):
        tokens = [replace.get((si, wi), w.text) for wi, w in enumerate(sent.words)]
        out.append(" ".join(tokens).replace(" .", ".").replace(" ,", ","))
    return " ".join(out)

print(resolve(doc))

Barack Obama was born in Hawaii. Barack Obama was elected president in 2008, and Barack Obama's wife Michelle joined Barack Obama in Washington. Apple unveiled a new phone on Tuesday. The company said a new phone would go on sale next month.


- 「Apple」と「The company」のように、代名詞だけでなく名詞句同士（言い換え）の同一性も解決される。`it` は Apple ではなく、発売される「a new phone」に正しく結び付けられ、そのまま置き換えられている。
- `her` は目的格と所有格が同形なので、この簡易な置換では区別できない。品詞解析 (`pos`) を併用すると判別できる。
- 置換後の文は不自然になることがある。人が読むためではなく、関係抽出や検索の前処理として使う。
- Stanza の共参照モデルは英語のほか、ドイツ語、フランス語、スペイン語など多数の言語に対応している（日本語は未対応なので、前項の KWJA を使う）。